# Part 2: Chunking Lab - Strategy Comparison

Compare 5 chunking strategies on real estate data:
1. **Semantic** - Sentence similarity grouping (spaCy)
2. **Fixed** - Token-based fixed size (tiktoken)
3. **Sliding** - Overlapping windows
4. **Parent-Child** - Hierarchical chunking
5. **Late** - Contextual embedding (JINA-inspired)

**Goal**: Find optimal strategy for retrieval quality vs. cost

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from prime_lands.config import load_config
from prime_lands.chunking.semantic import SemanticChunker
from prime_lands.chunking.fixed import FixedChunker
from prime_lands.chunking.sliding import SlidingChunker
from prime_lands.chunking.parent_child import ParentChildChunker
from prime_lands.chunking.late import LateChunker
from prime_lands.crawler.models import PropertyDocument
from prime_lands.logger import setup_logger

setup_logger(level="INFO")
print("✓ Imports successful")

## Step 1: Load Data & Config

In [ ]:
cfg = load_config(Path.cwd().parent / "config.yaml")
data_dir = Path.cwd().parent / "data"

# Load crawled properties
corpus_file = data_dir / "primelands_corpus.jsonl"
properties = []

with open(corpus_file, "r", encoding="utf-8") as f:
    for line in f:
        properties.append(PropertyDocument(**json.loads(line)))

print(f"Loaded {len(properties)} properties")
print(f"Sample property: {properties[0].title}")

## Step 2: Initialize All Chunkers

In [ ]:
chunkers = {
    "semantic": SemanticChunker(cfg.chunking.semantic),
    "fixed": FixedChunker(cfg.chunking.fixed),
    "sliding": SlidingChunker(cfg.chunking.sliding),
    "parent_child": ParentChildChunker(cfg.chunking.parent_child),
    "late": LateChunker(cfg.chunking.late),
}

print("✓ Initialized 5 chunking strategies")

## Step 3: Chunk Sample Documents

In [ ]:
# Use first 10 properties for comparison
sample_properties = properties[:10]

all_results = {}

for strategy_name, chunker in chunkers.items():
    print(f"\nChunking with {strategy_name}...")
    all_chunks = []
    
    for prop in sample_properties:
        rag_text = prop.to_rag_text()
        chunks = chunker.chunk(rag_text, source_id=prop.property_id)
        all_chunks.extend(chunks)
    
    stats = chunker.get_stats(all_chunks)
    all_results[strategy_name] = {
        "chunks": all_chunks,
        "stats": stats,
    }
    
    print(f"  → {stats['total_chunks']} chunks created")
    print(f"  → Avg length: {stats.get('avg_length', stats.get('avg_chars', 0)):.0f} chars")

## Step 4: Compare Strategies

In [ ]:
# Create comparison dataframe
comparison = []

for strategy_name, result in all_results.items():
    stats = result["stats"]
    comparison.append({
        "Strategy": strategy_name,
        "Total Chunks": stats["total_chunks"],
        "Avg Chars": stats.get("avg_length", stats.get("avg_chars", 0)),
        "Avg Words": stats.get("avg_words", 0),
        "Avg Tokens": stats.get("avg_tokens", stats.get("avg_chunk_tokens", 0)),
    })

comparison_df = pd.DataFrame(comparison)
print("\n📊 Chunking Strategy Comparison:")
comparison_df

In [ ]:
# Visualize
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chunk count comparison
axes[0].bar(comparison_df["Strategy"], comparison_df["Total Chunks"])
axes[0].set_title("Total Chunks per Strategy")
axes[0].set_xlabel("Strategy")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=45)

# Average size comparison
axes[1].bar(comparison_df["Strategy"], comparison_df["Avg Chars"])
axes[1].set_title("Average Chunk Size (characters)")
axes[1].set_xlabel("Strategy")
axes[1].set_ylabel("Characters")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(Path.cwd().parent / "outputs" / "chunking_comparison.png", dpi=150)
plt.show()

## Step 5: Examine Sample Chunks

In [ ]:
# Show first chunk from each strategy
for strategy_name, result in all_results.items():
    chunk = result["chunks"][0]
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy_name.upper()}")
    print(f"Chunk length: {len(chunk.text)} chars")
    print(f"Metadata: {chunk.metadata}")
    print(f"\nContent preview:")
    print(chunk.text[:300] + "...")

## Step 6: Save Comparison Results

In [ ]:
# Save to CSV
outputs_dir = Path.cwd().parent / "outputs"
outputs_dir.mkdir(exist_ok=True)

comparison_df.to_csv(outputs_dir / "chunking_comparison.csv", index=False)
print(f"✓ Saved comparison to {outputs_dir / 'chunking_comparison.csv'}")

# Save chunks for best strategy (choose based on analysis)
# For now, let's use semantic as default
best_strategy = "semantic"
chunks_dir = data_dir / "chunks"
chunks_dir.mkdir(exist_ok=True)

# Chunk all properties with best strategy
best_chunker = chunkers[best_strategy]
all_chunks = []

for prop in properties:
    rag_text = prop.to_rag_text()
    chunks = best_chunker.chunk(rag_text, source_id=prop.property_id)
    all_chunks.extend(chunks)

# Save chunks as JSONL
chunks_file = chunks_dir / f"{best_strategy}_chunks.jsonl"
with open(chunks_file, "w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(chunk.model_dump_json() + "\n")

print(f"✓ Saved {len(all_chunks)} chunks to {chunks_file}")

---

## ✅ Part 2 Complete!

**Key Findings:**
- Compare chunk counts vs. quality trade-offs
- Note which strategy creates most/least chunks
- Consider retrieval precision vs. context completeness

**Next Steps:**
1. Proceed to `03_intelligence_layers.ipynb` to build RAG/CAG/CRAG
2. Document strategy selection rationale in engineering report